# Lab 4. Data analysis workshop

Igor Gitelman  v0.3

In this lab we will use a Jupyter notebook as an experimental logbook. That means the notebook will contain not only code, but also short text explanations. You can answer the text parts in a markdown (text) cell — it does not have to be code.

In this lab you will:

  1. **Using the notebook as a logbook tool:** Learn how to structure your notebook so that it serves as a real experimental record — combining code, results, plots, and short explanations in one document.
  
  2. **Weighted average and uncertainty handling**: Learn how to combine repeated measurements with different errors using the weighted average formula

  3. **Work with uncertainties**: fit the same data with several models and use the measurement errors as weights. Then we will evaluate the fits using reduced chi-squared $\chi^2_\nu$. This shows how error bars affect the result, not just the plot.

So: first write, then code, then interpret — all in one notebook.

In [26]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

<h1>Average the data</h1>

Let's prepare a measured data set of length $N=10$.

This means we measure the desired quantity ten times using the same method.

In [27]:
mu=1;
sigma=2;

N = 10 # number of measurements

X = mu + sigma * np.random.randn( N )
print("The first ten measurements:", X[:10])

The first ten measurements: [-0.49279854  0.6568815   0.58967176  1.87774051  2.00086902  0.53706321
  2.1066723  -0.41973572 -1.53386084  1.77460718]


Now let's calculate their average and standard deviation.

<p> Using the known formulas: </p>

$$\langle X \rangle = \frac{\sum X_i}{N},\,\,\, STD(X) = \sqrt{\frac{1}{N-1} \left(\sum (X_i-\langle X \rangle)^2 \right)} $$
<p> Here, \( N \) is the number of measurements (the length of the measurement array). Pay attention to Bessel’s correction. </p>

In [28]:
# TODO: replace np.nan with your calculation use numpy mean and std(x,ddof=1) 

mu_X = np.nan
std_X = np.nan

print(f"The measured value: {mu_X:.3f}")
print(f"The standard deviation: {std_X:.3f}")

The measured value: nan
The standard deviation: nan


* Try varying N, the number of measurements.
* Do the measured value and standard deviation change?
* Why?

The uncertainty of the average is

$s=\frac{STD(X)}{\sqrt{N}}$
 * Compute in Python
 * Write as ‘value ± uncertainty’.
 * Write the physical relative uncertainty (error) $\frac{\text{uncertainty}}{\text{value}}$
 * Does the number of measurements affect the relative uncertainty of the average?

In [29]:
unc_mean = np.nan
rel_unc = np.nan

print(f"Measured value: {mu_X:.3f} ± {unc_mean:.3f}")
print(f"Relative uncertainty: {rel_unc:.3f}")

Measured value: nan ± nan
Relative uncertainty: nan


# Weighted average

When measurements have different uncertainties, they do not contribute equally to the estimate of the true value.
A result with a smaller uncertainty is statistically more reliable and should therefore influence the average value more strongly than a less precise one.

The weighted average incorporates this by assigning each measurement a weight proportional to the inverse of its variance, $w_i = \frac{1}{\sigma_i^2}$.
This means that precise measurements (small $\sigma_i$) have larger weights, while uncertain ones contribute less.

Mathematically, the best estimate of the average is

$$\langle X \rangle_w=\frac{\sum_i w_i X_i}{\sum_i w_i}, \,\,\, \sigma _{\langle X \rangle_w} =\frac{1}{\sqrt{\sum_i w_i}}$$

And the final result would be

$$ X=\langle X \rangle_w \pm \sigma _{\langle X \rangle_w}$$

This approach ensures that all measurements are combined consistently, reflecting both their values and their statistical confidence.

## Step 1 — Start with simple average

<p> You measured the same physical quantity three times:</p>



```
x=[1.05,0.97,1.10]
```

* Compute the simple (unweighted) average using `np.mean()`.


In [32]:
x = np.array([1.05, 0.97, 1.10])
# TODO: compute the simple average
mu_simple = np.nan

## Step 2 — Notice the problem
<p>Now suppose you also know the uncertainties of each measurement: </p>

`sigma = np.array([0.10, 0.30, 0.20])`

* Task: print the values and their uncertainties in one line each. ($A\pm \delta a$)
* Question: are these three measurements equally reliable? Compare their relative uncertainties, $\delta a/A$

## Step 3 — Ask: Should all of them count equally?

If one measurement has uncertainty 0.3 and another has uncertainty 0.1, should they affect the final answer equally?

* answer “yes” or “no” and explain in one sentence.

answer:

## Step 4 — Introduce weights

Define the weight of each measurement as

$$w_i=1/\sigma_i^2$$

Smaller uncertainty means a more reliable measurement.

Statistics uses the inverse of the variance because a precise measurement contains more information, so it should count more.

A noisy measurement (large $\sigma$) is less trustworthy and therefore receives a smaller weight.


* Task: in Python, compute the array of weights from the array of uncertainties.
* Task: print the weights and see which measurement got the biggest weight.

## Step 5 — Build the weighted average formula from parts

Using the weights, form the two sums

$$ S_1=\sum_i w_i x_i ,\,\,\,\, S_2=\sum_i w_i.$$


* Task: compute S1 and S2 in Python.
* Task: print both.

## Step 6 — Get the weighted average

Define the weighted average as

$$ \langle x \rangle_w = \frac{S_1}{S_2}$$

* Task: compute it in Python.

* Task: print both the simple average (from step one) and the weighted average and compare.


## Step 7 — Uncertainty of the weighted average

The uncertainty of the weighted average is

$\sigma_{
\langle x \rangle_w} = \frac{1}{\sqrt{\sum_i w_i}}$

* Task: compute this in Python.
* Task: print the result as “value ± uncertainty”.

## Step 8 — Test the method

Add a very bad measurement, for example $x_4=0.5 \pm 5.0$

* Task: append it to both arrays.
* Task: recompute the simple average and the weighted average.
* Question: which average changed more? What does it tell you?

## Step 9 — Special case check

Set all uncertainties to be the same, for example ` sigma = [0.2,0.2,0.2,0.2] `

* Task: recompute the weighted average.
* Question: is it now equal to the simple average?


## Step 10 — Wrap it into a function

Write a function


```
def weighted_average(values, errors):
    ...
```
that:

* checks that values and errors have the same length,
* computes the weighted average,
* computes its uncertainty,
* returns both.



# Reduced $\chi^2$ (Chi-Square) Analysis

When we compare measured data to a theoretical model, we need to know not only how close the data are to the model but also whether the differences are statistically reasonable given the measurement uncertainties.

The chi-square test does this by summing all squared deviations weighted by their uncertainties:

$$\chi^2 = \sum_i \frac{(y_i-y_{i,model})^2}{\sigma_i^2}$$

To compare fits with different numbers of data points or free parameters, we use the reduced chi-square,

$$ \chi_\nu^2=\frac{\chi^2}{\nu}, \,\,\,\, \nu = N_{\text{data}}-N_{\text{fit parameters}}$$

In reduced chi-squared, the **degrees of freedom** $\nu$ is the number of data points minus the number of fitted parameters, i.e. how many points are “left over” to test the model after you’ve used some of them to determine the parameters.

* If $\chi_\nu^2 \approx 1$, the model agrees with the data within experimental uncertainty;
* $\chi_\nu^2 \gg 1$ means that the scatter is larger than expected (poor fit or underestimated errors),
* and $\chi_\nu^2 \ll 1$ means that the data are too close (overestimated errors or overfitting).


## Step 1 — Create sample data


```
import numpy as np
x  = np.array([1, 2, 3, 4, 5, 6])
y  = np.array([3.1, 5.0, 7.2, 9.1, 12.0, 13.9])
sy = np.array([0.15, 0.15, 0.2, 0.2, 0.25, 0.25])  # measurement errors
```

* Task: print all arrays; check that each data point has its own uncertainty.

## Step 2 — Define a simple model and fit


``` python
from scipy.optimize import curve_fit

def model(x,a):
    return a*x
parameters, _ = curve_fit(model, x, y, sigma=sy, absolute_sigma=True)
```

to fit a line with weights $1/\sigma_i^2$.

We use `absolute_sigma=True` because `sy` represents the actual experimental uncertainties of the measured data points. Therefore, the absolute size of the error bars matters for the parameter uncertainties and for the interpretation of chi-square. 

* Task: get the best-fit slope a_fit, predict y_fit.

## Step 3 — Compute the Fit Quality with Chi-Squared statistics

Compute the chi-squared $\chi^2$, the number of degrees of freedom $\nu = N_\text{data} - N_\text{params}$,
and the reduced chi-squared $\chi^2_\nu = \chi^2 / \nu$ to evaluate how well your model fits the data.

```
d=y.size-parameters.size
chi2 = np.sum(((y - model(x, a_fit)) / sy)**2)
print("Chi-square =", chi2)
chi2_red = chi2 / d
print("Reduced chi-square =", chi2_red)
```
* Task: explain what each term in the formula represents (difference, weight, square).
* Task: interpret if the result is ≈ 1, ≫ 1, or ≪ 1.

## Step 4 — Test sensitivity
Increase all uncertainties by a factor of 10.

* Task: recompute $\chi_\nu^2$. 
* By what factor should reduced chi-square change?

* Does this indicate underestimated or overestimated uncertainties? Explain.

## Step 5 — Add parameters
Fit $y=ax+b$ (two parameters).

* Task: update $ν = N - 2$, recalculate $\chi_\nu^2$, and discuss how degrees of freedom change.

In [ ]:

def model_linear(x, a, b):
    return a*x + b

# TODO: use curve_fit with model_linear and remember that this model has 2 parameters

## Step 6 — Compare visually
Plot data with error bars and model line.

```
import matplotlib.pyplot as plt
plt.errorbar(x, y, yerr=sy, fmt='o', label='Data')
plt.plot(x, model(x, a_fit), label='Fit')
plt.legend(); plt.show()
```
* Task: by eye, does the fit quality agree with your $\chi_\nu^2$ interpretation?

## Step 7 — Write a function
Objective:
Write a Python function that calculates the chi-squared $\chi^2$ and reduced chi-squared $\chi^2_\nu$ values for a given model fit.

```
def chi2_reduced(y, y_model, errors, n_params):
    ...

```

that returns both $\chi^2$ and $\chi_\nu^2$.

* Task: Test it on datasets using three different models:
  * $y = a x$
  * $y = ax + b$
  * $y = ax^2 + bx + c$
* Plot!

This means fitting each model to the data points using **curve_fit**, and then calculating the chi-squared $\chi^2$ and reduced chi-squared $\chi^2_\nu$ using that function.

Adding more parameters can reduce chi-square, but it also reduces the number of degrees of freedom. A smaller reduced chi-square does not automatically mean a better physical model.


## Step 8 — Conclude

* If $\chi_\nu^2 \approx 1$ → good uncertainties and reasonable model.
* If $\chi_\nu^2 \gg 1$ → model missing physics or underestimated errors.
* If $\chi_\nu^2 \ll 1$ → overestimated uncertainties or overfitting.

Write one sentence explaining which case applies to your data.